# compile value-added catalog for DESI-ACT LRGs

In [1]:
import os
import glob
import torch
import numpy as np
from astropy.table import Table as aT
from astropy.cosmology import Cosmology, Planck13

In [2]:
from sedflow import flows as F

/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import corner as DFM
# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams['text.usetex'] = True
#mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

In [4]:
if torch.cuda.is_available(): device = 'cuda'
else: device = 'cpu'

# import flows
desiflow = F.DESIflow(name='modelb.lowz.grzW1W2', device=device)

In [5]:
lrgs = aT.read('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/overlapping_catalog_legacy_info.txt', 
               format='ascii')

In [6]:
columns = ['TARGETID', 'RA', 'DEC', "Z", "FLUX_G", "FLUX_R", "FLUX_Z", "FLUX_W1", "FLUX_W2", 
           "MW_TRANSMISSION_G", "MW_TRANSMISSION_R", "MW_TRANSMISSION_Z", "MW_TRANSMISSION_W1", "MW_TRANSMISSION_W2",
           "FLUX_IVAR_G", "FLUX_IVAR_R", "FLUX_IVAR_Z", "FLUX_IVAR_W1", "FLUX_IVAR_W2"]

for i in range(len(lrgs.colnames)): 
    lrgs.rename_column('col%i' % (i+1), columns[i])

In [ ]:
lrgs['logmstar_median'] = np.repeat(-999., len(lrgs))
lrgs['sigma_logmstar'] = np.repeat(-999., len(lrgs))
for igal in np.arange(len(lrgs)): 
    fpost = os.path.join('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act',
                         'desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.%i.npy' % igal)
    if os.path.isfile(fpost): 
        try: 
            post = np.load(fpost)
        except: 
            print('%s has issues' % fpost)
        tage = Planck13.age(lrgs['Z'][igal]).value # age in Gyr
        
        # below is to correct for the fact that I forgot to switch log10(Z) to Z
        post[:,7:9] = 10**post[:,7:9] 

        # calculate surviving stellar mass
        logmstar = desiflow._msurv(post, np.repeat(tage, post.shape[0]))
        
        lrgs['logmstar_median'][igal] = np.median(logmstar)
        lrgs['sigma_logmstar'][igal] = np.std(logmstar)
    else: 
        print('%s does not exist' % fpost)
        
    if igal % 10000:
        lrgs.write('desi_act_overlap.lrg.vagc.hdf5', format='hdf5')

/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.10.npy has issues
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.56.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.293.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.522.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.1009.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.1165.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.1257.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/l

/global/u1/c/chahah/projects/SEDflow/src/sedflow/flows.py:341: RuntimeWarning: invalid value encountered in divide
  tt_u[:,i] = 1. - (tt[:,i] / np.prod(tt_u[:,:i], axis=1))


/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.9921.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.9932.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.10095.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.10259.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.10327.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.10965.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.11118.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/s

/global/u1/c/chahah/projects/SEDflow/src/sedflow/flows.py:381: RuntimeWarning: divide by zero encountered in log10
  logmsurv = thetas[:,0] + np.log10(fsurv)
/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/numpy/core/_methods.py:173: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.43629.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.43766.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.45603.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.45668.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.45899.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.47140.npy does not exist
/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.48360.npy does not exist
/global/cfs/projectdirs/desi/users/chahah

In [15]:
lrgs.write('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desi_act_overlap.lrg.vagc.hdf5', format='hdf5')

In [18]:
for fpost in glob.glob('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1_della/*npy'): 
    igal = int(fpost.split('.')[-2])
    try: 
        post = np.load(fpost)
    except: 
        print('%s has issues' % fpost)
    tage = Planck13.age(lrgs['Z'][igal]).value # age in Gyr

    # below is to correct for the fact that I forgot to switch log10(Z) to Z
    post[:,7:9] = 10**post[:,7:9] 

    # calculate surviving stellar mass
    logmstar = desiflow._msurv(post, np.repeat(tage, post.shape[0]))

    lrgs['logmstar_median'][igal] = np.median(logmstar)
    lrgs['sigma_logmstar'][igal] = np.std(logmstar)

/global/u1/c/chahah/projects/SEDflow/src/sedflow/flows.py:341: RuntimeWarning: invalid value encountered in divide
  tt_u[:,i] = 1. - (tt[:,i] / np.prod(tt_u[:,:i], axis=1))


In [25]:
lrgs.write('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/desi_act_overlap.lrg.vagc.hdf5', format='hdf5', overwrite=True)